# B3 — Chargement de conteneurs : Bin Packing 3D

**Emile Jouannet — EPITA SCIA 2026**

---

## Problème

Le **Bin Packing 3D** consiste à placer un ensemble d'objets parallélépipédiques dans un nombre **minimal** de conteneurs identiques, en respectant :
- les dimensions du conteneur (largeur W, profondeur D, hauteur H)
- le **non-chevauchement** des objets
- les contraintes de **fragilité** (objets fragiles ne peuvent pas supporter d'autres objets)
- le **poids maximum** par conteneur

C'est un problème NP-difficile — généralisation 3D du bin packing classique. On le résout ici avec **Google OR-Tools CP-SAT**, comparé à une heuristique de référence (First Fit Decreasing).

## 1. Installation et imports

In [ ]:
import sys
import os
import time

# notebook/ est un niveau en dessous de la racine du projet
sys.path.insert(0, os.path.abspath('..'))

from src.model import Item, Container, solve
from src.heuristic import first_fit_decreasing
from src.visualization import plot_bin, plot_all_bins, summary_table

print('Imports OK')

## 2. Modélisation CP-SAT

### Variables de décision

Pour $n$ objets et au plus $n$ conteneurs :

| Variable | Domaine | Signification |
|----------|---------|---------------|
| `bin[i]` | $[0, n-1]$ | indice du conteneur de l'objet $i$ |
| `x[i]` | $[0, W - w_i]$ | position en X de l'objet $i$ |
| `y[i]` | $[0, D - d_i]$ | position en Y |
| `z[i]` | $[0, H - h_i]$ | position en Z |

### Contrainte de non-chevauchement

Pour chaque paire $(i, j)$ dans le **même conteneur**, au moins une des 6 séparations axiales doit tenir :

$$x_i + w_i \leq x_j \quad \text{OU} \quad x_j + w_j \leq x_i \quad \text{OU} \quad y_i + d_i \leq y_j \quad \ldots$$

On encode ça avec des variables booléennes `sep[k]` et `OnlyEnforceIf`.

### Cassage de symétrie

Sans cassage, permuter deux conteneurs vides donne la même solution → explosion combinatoire. On impose `bin[0] = 0` et `bin[i] <= bin[i-1] + 1`.

### Objectif

$$\text{Minimiser} \quad \max_i(\text{bin}[i]) + 1$$

## 3. Exemple simple — 3 objets, 1 conteneur

In [ ]:
container = Container(W=10, D=10, H=10, max_weight=100.0)

items_simple = [
    Item(w=3, d=4, h=5, weight=10.0),
    Item(w=6, d=2, h=3, weight=15.0),
    Item(w=4, d=4, h=4, weight=12.0),
]

result = solve(items_simple, container, time_limit=30)

print(f"Statut     : {result['status']}")
print(f"Conteneurs : {result['num_bins']}")
print(f"Temps      : {result['solve_time']:.3f}s")
print()
for i, (item, b, pos) in enumerate(zip(items_simple, result['assignment'], result['positions'])):
    print(f"  Objet {i} ({item.w}x{item.d}x{item.h}) → conteneur {b}, position {pos}")

### Visualisation 3D du résultat

In [ ]:
fig = plot_bin(items_simple, result, bin_idx=0, container=container)
fig.show()

## 4. Exemple avec fragilité et poids

In [ ]:
container_small = Container(W=10, D=10, H=8, max_weight=50.0)

items_fragile = [
    Item(w=4, d=4, h=2, weight=5.0,  fragile=True),   # objet fragile (en haut)
    Item(w=4, d=4, h=3, weight=20.0, fragile=False),  # objet lourd
    Item(w=3, d=3, h=3, weight=15.0, fragile=False),  # objet normal
    Item(w=5, d=3, h=2, weight=8.0,  fragile=False),  # objet normal
]

result_fragile = solve(items_fragile, container_small, time_limit=30)

print(f"Statut     : {result_fragile['status']}")
print(f"Conteneurs : {result_fragile['num_bins']}")
print(f"Temps      : {result_fragile['solve_time']:.3f}s")
print()
for i, (item, b, pos) in enumerate(zip(items_fragile, result_fragile['assignment'], result_fragile['positions'])):
    tag = ' [FRAGILE]' if item.fragile else ''
    print(f"  Objet {i}{tag} ({item.w}x{item.d}x{item.h}, {item.weight}kg) → conteneur {b}, z={pos[2]}")

In [ ]:
fig = plot_bin(items_fragile, result_fragile, bin_idx=0, container=container_small)
fig.show()

## 5. Comparaison CP-SAT vs Heuristique FFD

L'heuristique **First Fit Decreasing (FFD)** :
1. Trie les objets par volume décroissant
2. Place chaque objet dans le premier conteneur où il entre

Rapide (O(n log n)) mais sous-optimale. CP-SAT trouve l'optimum mais est plus lent.

In [ ]:
import random
random.seed(42)

def generate_items(n, container, seed=42):
    random.seed(seed)
    items = []
    for _ in range(n):
        w = random.randint(1, container.W // 2)
        d = random.randint(1, container.D // 2)
        h = random.randint(1, container.H // 2)
        weight = random.uniform(1, 10)
        items.append(Item(w=w, d=d, h=h, weight=weight))
    return items

container_bench = Container(W=10, D=10, H=10)

print(f"{'N objets':<10} {'CP-SAT bins':<14} {'FFD bins':<12} {'CP-SAT temps':<15} {'Gain':<8}")
print("-" * 60)

for n in [3, 5, 8, 10, 12]:
    items = generate_items(n, container_bench)
    
    # CP-SAT
    t0 = time.time()
    res_cp = solve(items, container_bench, time_limit=30)
    t_cp = time.time() - t0
    
    # FFD
    res_ffd = first_fit_decreasing(items, container_bench)
    
    cp_bins = res_cp['num_bins'] if res_cp['num_bins'] else 'N/A'
    ffd_bins = res_ffd['num_bins']
    
    if isinstance(cp_bins, int):
        gain = ffd_bins - cp_bins
        gain_str = f"+{gain}" if gain > 0 else str(gain)
    else:
        gain_str = 'N/A'
    
    print(f"{n:<10} {str(cp_bins):<14} {ffd_bins:<12} {t_cp:<15.3f} {gain_str:<8}")

## 6. Benchmark sur instances générées — analyse de scalabilité

In [ ]:
import plotly.graph_objects as go

sizes = [3, 5, 7, 8, 10]
cp_times = []
cp_bins_list = []
ffd_bins_list = []

for n in sizes:
    items = generate_items(n, container_bench, seed=0)
    
    t0 = time.time()
    res_cp = solve(items, container_bench, time_limit=30)
    cp_times.append(time.time() - t0)
    cp_bins_list.append(res_cp['num_bins'] if res_cp['num_bins'] else None)
    
    res_ffd = first_fit_decreasing(items, container_bench)
    ffd_bins_list.append(res_ffd['num_bins'])

fig = go.Figure()
fig.add_trace(go.Bar(name='CP-SAT (optimal)', x=sizes, y=cp_bins_list, marker_color='#3cb44b'))
fig.add_trace(go.Bar(name='FFD (heuristique)', x=sizes, y=ffd_bins_list, marker_color='#e6194b'))
fig.update_layout(
    title='Nombre de conteneurs : CP-SAT vs FFD',
    xaxis_title='Nombre d\'objets',
    yaxis_title='Conteneurs utilisés',
    barmode='group',
    template='plotly_white'
)
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=sizes, y=cp_times, mode='lines+markers', name='Temps CP-SAT (s)', marker_color='#4363d8'))
fig2.update_layout(
    title='Temps de résolution CP-SAT selon la taille',
    xaxis_title='Nombre d\'objets',
    yaxis_title='Temps (s)',
    template='plotly_white'
)
fig2.show()

## 7. Démo finale — instance réaliste

Un camion 10x10x10 avec 8 colis de tailles variées, dont 2 fragiles.

In [ ]:
camion = Container(W=10, D=10, H=10, max_weight=100.0)

colis = [
    Item(w=4, d=4, h=3, weight=15.0, fragile=False),  # caisse lourde
    Item(w=3, d=3, h=2, weight=5.0,  fragile=True),   # colis fragile
    Item(w=5, d=2, h=4, weight=20.0, fragile=False),  # caisse lourde
    Item(w=2, d=2, h=2, weight=3.0,  fragile=True),   # colis fragile petit
    Item(w=4, d=3, h=3, weight=12.0, fragile=False),  # caisse moyenne
    Item(w=3, d=5, h=2, weight=8.0,  fragile=False),  # caisse plate
    Item(w=2, d=3, h=5, weight=10.0, fragile=False),  # colis haut
    Item(w=4, d=2, h=3, weight=9.0,  fragile=False),  # caisse normale
]

print("Résolution en cours...")
result_demo = solve(colis, camion, time_limit=60)

stats = summary_table(colis, result_demo, camion)
print(f"\nStatut          : {stats['status']}")
print(f"Objets          : {stats['num_items']}")
print(f"Conteneurs      : {stats['num_bins']}")
print(f"Efficacite vol. : {stats['space_efficiency_pct']}%")
print(f"Temps solve     : {stats['solve_time_s']:.3f}s")

print("\nAffectations :")
for i, (item, b, pos) in enumerate(zip(colis, result_demo['assignment'], result_demo['positions'])):
    tag = ' [FRAGILE]' if item.fragile else ''
    print(f"  Colis {i}{tag} ({item.w}x{item.d}x{item.h}) → camion {b}, pos={pos}")

In [ ]:
figures = plot_all_bins(colis, result_demo, camion)
for i, fig in enumerate(figures):
    print(f"--- Camion {i} ---")
    fig.show()

## 8. Limites et perspectives

### Limites du modèle actuel

- **Scalabilité** : les contraintes de non-chevauchement sont en $O(n^2)$ — au-delà de ~15 objets, CP-SAT devient lent (problème NP-difficile)
- **Orientation fixe** : les objets ne peuvent pas être retournés (pas de rotation)
- **Fragilité simplifiée** : on interdit tout objet au-dessus d'un objet fragile (même sans contact)

### Extensions possibles

- **Rotations** : ajouter des variables booléennes pour autoriser la rotation (6 orientations par objet)
- **Grandes instances** : utiliser une heuristique ALNS (Adaptive Large Neighborhood Search) ou coupler avec la formulation de Martello & Vigo
- **Stabilité** : ajouter une contrainte sur le centre de gravité par couche
- **Multi-type de conteneurs** : conteneurs de tailles différentes (camions, palettes)

### Conclusion

CP-SAT trouve la solution **optimale** sur de petites instances (≤10 objets) et s'avère meilleur que FFD sur les cas difficiles. Pour les grandes instances, l'approche hybride CP + heuristique est la voie standard dans la littérature (Crainic et al., 2008).